In [ ]:
# --- Step 1: Import dependencies ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Step 2: Load your data ---
sequences_df = pd.read_csv('/content/la matrice sequences.csv')
rgb_df = pd.read_csv('/content/la matrice.csv')

# --- Step 3: Build color-to-RGB mapping ---
real_color_to_rgb = {
    row['color'].strip().lower(): (row['r'], row['g'], row['b'])
    for _, row in rgb_df.iterrows()
}

# --- Step 4: Define low-pass filter ---
def low_pass_filter(signal, alpha=0.3):
    smoothed = [signal[0]]
    for i in range(1, len(signal)):
        smoothed.append(alpha * signal[i] + (1 - alpha) * smoothed[-1])
    return np.array(smoothed)

# --- Step 5: Define interactive analysis function ---
def analyze_sequence(sequence_name):
    clear_output(wait=True)
    seq_row = sequences_df[sequences_df['name'] == sequence_name].iloc[0]
    seq_colors = [color.strip().lower() for color in seq_row['sequence'].split(',')]
    seq_rgb = np.array([real_color_to_rgb.get(color, (0, 0, 0)) for color in seq_colors])

    smoothed_r = low_pass_filter(seq_rgb[:, 0])
    smoothed_g = low_pass_filter(seq_rgb[:, 1])
    smoothed_b = low_pass_filter(seq_rgb[:, 2])
    smoothed_rgb = np.stack([smoothed_r, smoothed_g, smoothed_b], axis=1)

    derivatives = np.diff(smoothed_rgb, axis=0)

    plt.figure(figsize=(12, 8))
    plt.subplot(3, 1, 1)
    plt.plot(seq_rgb, marker='o')
    plt.title(f'Original RGB Sequence: {sequence_name}')
    plt.legend(['R', 'G', 'B'])

    plt.subplot(3, 1, 2)
    plt.plot(smoothed_rgb, marker='o')
    plt.title('Smoothed RGB Sequence (Low-pass Filter)')
    plt.legend(['R', 'G', 'B'])

    plt.subplot(3, 1, 3)
    plt.plot(derivatives, marker='o')
    plt.title('Derivative of Smoothed RGB Sequence')
    plt.legend(['dR/dt', 'dG/dt', 'dB/dt'])

    plt.tight_layout()
    plt.show()

# --- Step 6: Launch the interactive dropdown ---
dropdown = widgets.Dropdown(
    options=sequences_df['name'].unique(),
    description='Sequence:',
    disabled=False,
)

widgets.interact(analyze_sequence, sequence_name=dropdown)

interactive(children=(Dropdown(description='Sequence:', options=('plot', 'knot', 'pain', 'practical', 'spiritu…

<function __main__.analyze_sequence(sequence_name)>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.signal import butter, filtfilt, savgol_filter

# Load your data
sequences_df = pd.read_csv('/content/la matrice sequences.csv')
rgb_df = pd.read_csv('/content/la matrice.csv')

# Build color-to-RGB mapping
real_color_to_rgb = {
    row['color'].strip().lower(): (row['r'], row['g'], row['b'])
    for _, row in rgb_df.iterrows()
}

# Define filtering functions
def butter_lowpass_filter(data, cutoff=0.1, order=2):
    b, a = butter(order, cutoff, btype='low', analog=False)
    if len(data) > max(len(b), len(a)) * 3:
        return filtfilt(b, a, data)
    else:
        return data  # fallback: return raw if too short

def savgol_smooth(data, window_length=3, polyorder=2):
    if len(data) >= window_length:
        return savgol_filter(data, window_length=window_length, polyorder=polyorder)
    else:
        return data  # fallback: return raw if too short

# Define interactive analysis function
def analyze_sequence(sequence_name, method='butterworth'):
    clear_output(wait=True)
    seq_row = sequences_df[sequences_df['name'] == sequence_name].iloc[0]
    seq_colors = [color.strip().lower() for color in seq_row['sequence'].split(',')]
    seq_rgb = np.array([real_color_to_rgb.get(color, (0, 0, 0)) for color in seq_colors])

    if method == 'butterworth':
        smoothed_r = butter_lowpass_filter(seq_rgb[:, 0])
        smoothed_g = butter_lowpass_filter(seq_rgb[:, 1])
        smoothed_b = butter_lowpass_filter(seq_rgb[:, 2])
    elif method == 'savgol':
        smoothed_r = savgol_smooth(seq_rgb[:, 0], window_length=3)
        smoothed_g = savgol_smooth(seq_rgb[:, 1], window_length=3)
        smoothed_b = savgol_smooth(seq_rgb[:, 2], window_length=3)
    else:
        smoothed_r, smoothed_g, smoothed_b = seq_rgb[:, 0], seq_rgb[:, 1], seq_rgb[:, 2]

    smoothed_rgb = np.stack([smoothed_r, smoothed_g, smoothed_b], axis=1)
    derivatives = np.diff(smoothed_rgb, axis=0)

    plt.figure(figsize=(12, 8))
    plt.subplot(3, 1, 1)
    plt.plot(seq_rgb, marker='o')
    plt.title(f'Original RGB Sequence: {sequence_name}')
    plt.legend(['R', 'G', 'B'])

    plt.subplot(3, 1, 2)
    plt.plot(smoothed_rgb, marker='o')
    plt.title(f'Smoothed RGB Sequence ({method.title()} Filter)')
    plt.legend(['R', 'G', 'B'])

    plt.subplot(3, 1, 3)
    if derivatives.shape[0] > 0:
        plt.plot(derivatives, marker='o')
    else:
        plt.plot([0], [0], 'o')  # placeholder if too short
    plt.title('Derivative of Smoothed RGB Sequence')
    plt.legend(['dR/dt', 'dG/dt', 'dB/dt'])

    plt.tight_layout()
    plt.show()

# Set up interactive widgets
dropdown = widgets.Dropdown(
    options=sequences_df['name'].unique(),
    description='Sequence:',
    disabled=False,
)

method_dropdown = widgets.Dropdown(
    options=['butterworth', 'savgol', 'none'],
    value='butterworth',
    description='Filter:',
    disabled=False,
)

widgets.interact(analyze_sequence, sequence_name=dropdown, method=method_dropdown)


interactive(children=(Dropdown(description='Sequence:', options=('plot', 'knot', 'pain', 'practical', 'spiritu…

<function __main__.analyze_sequence(sequence_name, method='butterworth')>